In [ ]:
import pandas as pd
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
import matplotlib.colors as mcolors

from src.load_data import load_var, load_all_subjects

In [ ]:
def plot_pollutant_with_location_boundaries(data, pollutant='VOC', regime_filter='dynamic'):
    """
    Plots pollutant levels over time, marking location boundaries.

    Parameters:
        data (pd.DataFrame): DataFrame with at least 'object', 'regime', 'location', and the pollutant column.
        pollutant (str): The name of the pollutant column to plot.
        regime_filter (str): One of ['dynamic', 'static', 'all'] to control which regime to display.
    """
    # Validate regime filter
    valid_filters = ['dynamic', 'static', 'all']
    if regime_filter not in valid_filters:
        raise ValueError(f"regime_filter must be one of {valid_filters}")

    # Apply regime filter
    if regime_filter != 'all':
        filtered_data = data[data['regime'] == regime_filter].copy()
    else:
        filtered_data = data.copy()

    # Create a time index per object
    filtered_data['time_index'] = filtered_data.groupby('object').cumcount()

    # Plot each object separately
    for obj in filtered_data['object'].unique():
        subset = filtered_data[filtered_data['object'] == obj].copy()

        # Find where location changes
        subset['location_shift'] = subset['location'].ne(subset['location'].shift())
        location_change_indices = subset.index[subset['location_shift']].tolist()

        # Plotting
        plt.figure(figsize=(14, 6))
        sns.lineplot(data=subset, x='time_index', y=pollutant, hue='location', palette='Set2')

        for idx in location_change_indices:
            x_val = subset.loc[idx, 'time_index']
            loc_label = subset.loc[idx, 'location']
            plt.axvline(x=x_val, color='gray', linestyle='--', alpha=0.6)
            plt.text(x_val, subset[pollutant].max() * 0.95, f"{loc_label}",
                     rotation=90, verticalalignment='top', fontsize=8)

        plt.title(f'Object {obj}: {pollutant} Levels ({regime_filter.capitalize()} Regime)')
        plt.xlabel('Measurement Index')
        plt.ylabel(f'{pollutant} Level')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
voc_all = load_all_subjects('VOC')
voc_all.head()

In [ ]:
voc_1 = load_var(1, 'VOC')
voc_1['object'] = 1
voc_1.head()

In [ ]:
def filter_stabilized_dynamic(data, pollutant_col='VOC', window=10, std_threshold=2):
    """
    Filters out the initial spike in dynamic routes and keeps only the stabilized values.

    Parameters:
    - data: DataFrame containing columns ['object', 'location', 'regime', pollutant_col]
    - pollutant_col: the name of the column containing the pollutant level (default is 'VOC')
    - window: the size of the rolling window for statistics (default is 10)
    - std_threshold: the maximum standard deviation in the window to consider the values stable (default is 2)

    Returns:
    - DataFrame: filtered data with stabilized values only
    """
    dynamic_data = data[data['regime'] == 'dynamic']
    filtered = []

    grouped = dynamic_data.groupby(['object', 'location'])

    for (obj, loc), group in grouped:
        group = group.reset_index(drop=True)

        # Calculate rolling standard deviation for the pollutant values
        rolling_std = group[pollutant_col].rolling(window=window).std()

        # Find the first index where the rolling standard deviation is below the threshold
        stable_start = rolling_std[rolling_std < std_threshold].first_valid_index()

        if stable_start is not None:
            # Filter data starting from the point where stability is achieved
            group_filtered = group.iloc[stable_start + window - 1:]  # Shift by the size of the window
            filtered.append(group_filtered)

    # Concatenate all filtered groups and return
    return pd.concat(filtered, ignore_index=True)

In [ ]:
voc_filtered = filter_stabilized_dynamic(voc_1, pollutant_col='VOC', window=10, std_threshold=2)

In [ ]:
voc_filtered

In [ ]:
plot_pollutant_with_location_boundaries(voc_1, 'VOC', 'all')

In [ ]:
plot_pollutant_with_location_boundaries(voc_1, 'VOC', 'dynamic')

In [ ]:
plot_pollutant_with_location_boundaries(voc_filtered, 'VOC', 'dynamic')

In [ ]:
def compute_stability_thresholds(data, pollutant_col, last_fraction=0.3, window=10):
    """
    Automatically compute thresholds for std and diff based on the last part of the data,
    assuming it is already stable.
    """
    n = int(len(data) * last_fraction)
    tail = data.iloc[-n:].copy()

    rolling_std_tail = tail[pollutant_col].rolling(window).std()
    rolling_diff_tail = tail[pollutant_col].diff().abs()

    std_threshold = rolling_std_tail.median() + rolling_std_tail.std()
    diff_threshold = rolling_diff_tail.median() + rolling_diff_tail.std()

    return std_threshold, diff_threshold


def is_fully_stable(data, pollutant_col, std_threshold, diff_threshold, window):
    """
    Checks if the entire data sequence is already stable.
    """
    rolling_std = data[pollutant_col].rolling(window).std()
    rolling_diff = data[pollutant_col].diff().abs()

    return (rolling_std.dropna() < std_threshold).all() and (rolling_diff.dropna() < diff_threshold).all()


def find_dynamic_stable_start(data, pollutant_col='VOC', window=10, consecutive_windows=3):
    """
    Automatically detect the start index of the stable portion in a dynamic series.
    If the entire series is stable, returns 0.
    """
    std_threshold, diff_threshold = compute_stability_thresholds(data, pollutant_col, window=window)

    if is_fully_stable(data, pollutant_col, std_threshold, diff_threshold, window):
        return 0  # Entire segment is already stable

    rolling_std = data[pollutant_col].rolling(window).std()
    rolling_diff = data[pollutant_col].diff().abs()

    stable_flags = (rolling_std < std_threshold) & (rolling_diff < diff_threshold)

    for i in range(window - 1, len(data) - consecutive_windows):
        if stable_flags.iloc[i:i + consecutive_windows].all():
            return i + consecutive_windows

    return None

def filter_auto_stable_dynamic(data, pollutant_col='VOC', window=10, consecutive_windows=3):
    """
    Filters only the stable portion of dynamic data per object and location,
    using data-driven thresholds.
    """
    dynamic_data = data[data['regime'] == 'dynamic']
    filtered = []

    grouped = dynamic_data.groupby(['object', 'location'])

    for (obj, loc), group in grouped:
        group = group.reset_index(drop=True)

        stable_start = find_dynamic_stable_start(group, pollutant_col, window, consecutive_windows)

        if stable_start is not None:
            group_filtered = group.iloc[stable_start:]
            filtered.append(group_filtered)

    return pd.concat(filtered, ignore_index=True)

# Analyze VOC differences between locations (example for 1 object)

In [ ]:
voc_autofiltered = filter_auto_stable_dynamic(voc_1, window=10, pollutant_col='VOC')

Before filtering

In [ ]:
plot_pollutant_with_location_boundaries(voc_1, 'VOC', 'dynamic')

After filtering

In [ ]:
plot_pollutant_with_location_boundaries(voc_autofiltered, 'VOC', 'dynamic')

In [ ]:
def plot_pollutant_across_locations(data, pollutant_col='VOC'):

    plt.figure(figsize=(12, 6))
    sns.boxplot(x='location', y=pollutant_col, data=data)
    plt.title('VOC levels across locations (dynamic mode)')
    plt.xlabel('Location')
    plt.ylabel(pollutant_col)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# plot_voc_across_locations(voc_autofiltered, pollutant_col='VOC')

In [ ]:
def test_pollutant_across_locations(data, pollutant_col='VOC'):
    dynamic_data = data[data['regime'] == 'dynamic']
    groups = [group[pollutant_col].values for _, group in dynamic_data.groupby('location')]

    stat, p_value = kruskal(*groups)
    print(f"Kruskal-Wallis test statistic: {stat:.4f}, p-value: {p_value}")
    if p_value < 0.05:
        print("VOC levels differ significantly between locations.")
    else:
        print("No significant difference in VOC levels between locations.")

In [ ]:
test_pollutant_across_locations(voc_autofiltered)

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations
import statsmodels.stats.multicomp as mc

In [ ]:
def pairwise_with_statsmodels(data, pollutant_col='VOC'):
    comp = mc.MultiComparison(data[pollutant_col], data['location'])
    result = comp.allpairtest(mannwhitneyu, method='bonferroni')
    return result[0]

In [ ]:
pairwise_table = pairwise_with_statsmodels(voc_autofiltered)
pairwise_table

In [ ]:
data = pairwise_table.data[1:]
columns = ['group1', 'group2', 'stat', 'pval', 'pval_corr', 'reject']

pairwise_table = pd.DataFrame(data, columns=columns)
pairwise_table.head()

In [ ]:
def plot_pairwise_heatmap(df, pval_col='pval_corr'):

    pivot_table = df.pivot(index='group1', columns='group2', values=pval_col)

    plt.figure(figsize=(10, 8))
    heatmap = sns.heatmap(pivot_table, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=0, vmax=1)

    plt.title('Pairwise Comparison Heatmap', fontsize=16)
    plt.xlabel('Group 2', fontsize=12)
    plt.ylabel('Group 1', fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_pairwise_heatmap(pairwise_table)

# Analyze VOC differences between locations across all objects

In [ ]:
def analyze_locations_across_objects(num_objects=20, pollutant_col='VOC', window=10):
    all_voc = []

    # Step 1–2: Load and filter data for each object
    for i in range(1, num_objects + 1):
        voc_i = load_var(i, pollutant_col)
        voc_i['object'] = i
        voc_filtered = filter_auto_stable_dynamic(voc_i, window=window, pollutant_col=pollutant_col)
        all_voc.append(voc_filtered)

    # Step 3: Concatenate all filtered data
    voc_all = pd.concat(all_voc, ignore_index=True)

    # Step 4: Calculate mean VOC per object per location
    voc_grouped = voc_all.groupby(['object', 'location'])[pollutant_col].mean().reset_index()
    voc_pivot = voc_grouped.pivot(index='object', columns='location', values=pollutant_col)

    # Step 5: Plot heatmap of mean VOC levels per object/location
    plt.figure(figsize=(10, 6))
    sns.heatmap(voc_pivot, annot=True, cmap="YlOrRd", fmt=".1f")
    plt.title("Mean VOC per Object and Location")
    plt.xlabel("Location")
    plt.ylabel("Object")
    plt.tight_layout()
    plt.show()

    # Step 6: Plot individual object VOC and average line plot
    plt.figure(figsize=(12, 6))
    for obj in voc_pivot.index:
        plt.plot(voc_pivot.columns, voc_pivot.loc[obj], alpha=0.3, label=f'Object {obj}')

    # Mean line
    mean_values = voc_pivot.mean(axis=0)
    plt.plot(voc_pivot.columns, mean_values, color='black', label='Mean', linewidth=2)
    plt.title("VOC per Location across Objects")
    plt.xlabel("Location")
    plt.ylabel(f"Mean {pollutant_col}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return voc_pivot

In [ ]:
analyze_locations_across_objects()

In [ ]:
def remove_outlier_pvals(pvals, threshold=1.5):
    pvals = np.array(pvals)
    eps = 1e-10
    log_pvals = -np.log10(np.clip(pvals, eps, 1))

    median = np.median(log_pvals)
    mad = np.median(np.abs(log_pvals - median))
    if mad == 0:
        return pvals

    z_scores = np.abs(log_pvals - median) / (mad * 1.4826)
    return pvals[z_scores < z_thresh]

In [ ]:
def analyze_location_differences_across_objects(objects=range(1, 21), pollutant_col='VOC', remove_outlier=False):
    all_pvals = defaultdict(list)

    for obj_id in objects:
        # 1. Load and label data
        data = load_var(obj_id, pollutant_col)
        data['object'] = obj_id

        # 2. Filter dynamic stable data
        filtered = filter_auto_stable_dynamic(data, window=10, pollutant_col=pollutant_col)

        # 3. Compare location pairs
        result_table = pairwise_with_statsmodels(filtered, pollutant_col=pollutant_col)

        # 4. Parse result_table into DataFrame
        rows = result_table.data[1:]
        df = pd.DataFrame(rows, columns=['group1', 'group2', 'stat', 'pval', 'pval_corr', 'reject'])

        # 5. Convert values to numeric
        df['pval_corr'] = pd.to_numeric(df['pval_corr'], errors='coerce')

        # 6. Store values in both directions (symmetric)
        for _, row in df.iterrows():
            loc1 = row['group1']
            loc2 = row['group2']
            pval = row['pval_corr']
            if pd.notnull(pval):
                all_pvals[(loc1, loc2)].append(pval)
                all_pvals[(loc2, loc1)].append(pval)

    # 7. Get unique locations
    unique_locations = sorted({loc for pair in all_pvals.keys() for loc in pair})

    # 8. Create matrix with average pval_corr (без выбросов)
    heatmap_data = pd.DataFrame(index=unique_locations, columns=unique_locations, dtype=float)

    for (loc1, loc2), pvals in all_pvals.items():
        if remove_outlier:
            pvals = remove_outlier_pvals(pvals)
        if len(pvals) > 0:
            heatmap_data.loc[loc1, loc2] = np.mean(pvals)
        else:
            heatmap_data.loc[loc1, loc2] = np.nan

    cmap = mcolors.ListedColormap(["#67001f", "#f7f7f7"])
    bounds = [0, 0.05, 1.01]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    # 9. Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".3f",
        cmap=cmap,
        norm=norm,
        linewidths=0.5,
        linecolor='grey',
        cbar_kws={"label": "Average corrected p-value (without outliers)"}
    )
    plt.title("Average Corrected p-values Between Locations (Outliers Removed)")
    plt.tight_layout()
    plt.show()

    return heatmap_data

In [ ]:
analyze_location_differences_across_objects()

- GH is consistently significantly different from all others, indicating a unique environment or behavior.
- AB, BC, DE locations have moderate p-values with both Group 3 and each other. They likely represent transition areas or zones with variable conditions.
- CD, EF, FG locations show non-significant differences between each other, suggesting similar environmental characteristics.

In [ ]:
analyze_location_differences_across_objects(remove_outlier=True)

In [ ]:
analyze_location_differences_across_objects([2, 3, 4, 6, 7, 8, 9, 10, 11, 14, 15, 16, 17, 18, 19, 20])